In [ ]:
import sys
import glob
import torch
import json
import glob
import sys
import os
from sklearn.metrics import roc_curve, auc

import seaborn as sns

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from scipy.stats import norm
from collections import defaultdict
from IPython.display import clear_output

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats

from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

sys.path.append('../utils/')
sys.path.append('../src/model/')

sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")

from utils.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir
from utils.dprime import recompute_dprime_by_isi_per_subject
from utils.reliability import compute_itemwise_split_half_reliability

from utils.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utils.loading import load_results_with_exclusion_2
from utils.dprime import recompute_dprime_by_isi_per_subject
from utils.reliability import compute_itemwise_split_half_reliability
from utils.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utils.reliability import compute_power_curve
from utils.plotting import plot_power_curve

import DistanceMemoryModel
import encoders

def get_dprime_by_isi(df_per_subject, return_sem=False, return_subjects=False):
    """
    Compute mean d-prime per ISI across subjects, excluding ISI = -1 (lures).

    Args:
        df_per_subject (pd.DataFrame): Output from recompute_dprime_by_isi_per_subject.
        return_sem (bool): Whether to return standard error of the mean.
        return_subjects (bool): Whether to return per-subject d-primes too.

    Returns:
        pd.DataFrame or dict:
            If return_sem=False:
                DataFrame with columns ['isi', 'd_prime']
            If return_sem=True:
                DataFrame with columns ['isi', 'd_prime', 'sem']
            If return_subjects=True:
                Returns a dict with:
                    'summary': summary DataFrame as above,
                    'per_subject': filtered per-subject df
    """
    df_filtered = df_per_subject[df_per_subject["isi"] > -1]

    grouped = df_filtered.groupby("isi")["d_prime"]
    result_df = grouped.mean().reset_index(name="d_prime")

    if return_sem:
        result_df["sem"] = grouped.sem().values

    if return_subjects:
        return {
            "summary": result_df,
            "per_subject": df_filtered.copy()
        }

    return result_df.d_prime.tolist()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = device
print(device)
# grabbing example list of sound
sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/*wav")
texture_list = sounds_list

ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")
print(len(ALL_SOUNDS))

tasks = ["ind-nature-len120" ,"global-music-len120", "atexts-len120", "nhs-region-len120"]
which_task = tasks[2] # "global-music-len120", "atexts-len120" "nhs-region-len120"

base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"

seqs_paths = {"ind-nature-len120": "mem_exp_ind-nature_2025", 
              "global-music-len120": "global-music-2025-n_80",
              "atexts-len120": "mem_exp_atexts_2025",
              "nhs-region-len120": "nhs-region-n_80"}

hr_task_name = {"ind-nature-len120": "Industrial and Nature", 
              "global-music-len120": "Globalized Music",
              "atexts-len120": "Auditory Textures",
              "nhs-region-len120": " 'Natural History of Song' "}

In [ ]:
skip = False

if skip: 
    
    exps, seqs, fnames, _ = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                        min_dprime=2,
                                                        min_trials=120,
                                                        skip_len60=True,
                                                        verbose=False,
                                                        return_skipped=skip)
else:
    exps, seqs, fnames = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                    min_dprime=2,
                                                    min_trials=120,
                                                    skip_len60=True,
                                                    verbose=False,
                                                    return_skipped=skip)



move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

print("Number of participants used in analysis:", len(exps))


safe_name = which_task.lower().replace(" ", "_")  # e.g., "globalized_music"
save_dir = os.path.join("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only", safe_name)

# where you want the results
CSV_PATH = f"/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/"

ensure_dir(save_dir)
ensure_dir(CSV_PATH)
print(save_dir)

human_results = recompute_dprime_by_isi_per_subject(exps)
human_sensitivity = get_dprime_by_isi(human_results)

clear_output(wait=False)
CSV_PATH = f"/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/isi16_runs.csv"

from sklearn.metrics import mean_squared_error

# Your experiment structure (list of stimulus filepaths for each run)
experiment_list = []
for seq in seqs:
    list_to_add = []
    
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as file:
        data = json.load(file)
   
    for stim in  data['filenames_order']:
        edited_stim_name = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + stim
        list_to_add.append(edited_stim_name)
    
    experiment_list.append(list_to_add)

In [ ]:
# pc_dims = 256

# NV = 0.2                             # per-dim noise std per drift step (your class convention)
# CRIT_MULT = 1.5                          # criterion = CRIT_MULT * NV * sqrt(D)
# SEED = 123                               # set to None for stochastic runs
# DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# TIMES_TO_PLOT = [10, 17, 16, 22, 40, 80, 119]   # steps for hist panels

# # Map filenames to row indices in X0
# # 1) Collect all unique filenames in stable order
# seen, all_files = set(), []
# for seq in experiment_list:
#     for fn in seq:
#         if fn not in seen:
#             seen.add(fn)
#             all_files.append(fn)

# name_to_idx = {fn: i for i, fn in enumerate(all_files)}


# pc_texture_model = encoders.AudioTextureEncoderPCA(
#     statistics_dict=statistics_set.statistics,
#     pc_dims=pc_dims,
#     model_params=model_params,
#     sr=20000,
#     rms_level=0.05,
#     duration=2.0,
#     device=device
# )

# zscore_projector = encoders.ZScoreSpace(pc_texture_model, device=device)
# zscore_projector.fit(sounds_list)

# # 2) Use your pipeline to encode clean reps X0: [M,D]
# _tmp = DistanceMemoryModel.DistanceMemoryModel(
#     encoding_model=zscore_projector,
#     noise_variance=float(NV),   # irrelevant for encoding, but ctor requires it
#     criterion=0.0,
#     device=DEVICE
# )
# _tmp._fill_memory_bank(all_files)
# with torch.no_grad():
#     X0 = torch.stack([rep.detach().clone().view(-1) for rep in _tmp.memory_bank], dim=0).to(DEVICE)
# del _tmp
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

In [ ]:
# pick a common grid of FPR points
fpr_grid = np.linspace(0, 1, 100)

tprs = []
for exp in exps:
    a = np.array(exp.response, dtype=np.int64)
    b = np.array(exp.repeat == 'true', dtype=np.int64)
    fpr, tpr, _ = roc_curve(b, a, drop_intermediate=False)
    # interpolate each TPR onto the common FPR grid
    tprs.append(np.interp(fpr_grid, fpr, tpr))

tprs = np.array(tprs)
mean_human_tpr = tprs.mean(axis=0)
sem_human_tpr  = tprs.std(axis=0) / np.sqrt(len(exps))
human_auc_val = auc(fpr_grid, mean_human_tpr)
plt.plot(fpr_grid, mean_human_tpr, label=f"Mean Human ROC | AUC={human_auc_val:.3f}")
plt.fill_between(fpr_grid, mean_human_tpr-sem_human_tpr, mean_human_tpr+sem_human_tpr, alpha=0.3)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

In [ ]:
def _z(p, eps=1e-4):
    return norm.ppf(np.clip(p, eps, 1 - eps))
# Compute zROC from your mean ROC
z_fpr = _z(fpr_grid)
z_tpr = _z(mean_human_tpr)

# Fit straight line and quadratic on the finite region
ok = np.isfinite(z_fpr) & np.isfinite(z_tpr)
lin_coef  = np.polyfit(z_fpr[ok], z_tpr[ok], 1)      # slope, intercept
quad_coef = np.polyfit(z_fpr[ok], z_tpr[ok], 2)      # a, b, c  (zt ≈ a zf^2 + b zf + c)

slope, intercept = lin_coef
a, b, c = quad_coef

print(f"zROC linear slope = {slope:.3f}, intercept = {intercept:.3f}")
print(f"zROC curvature a = {a:.3e}  (concave-up if a>0)")

# Plot the zROC with the linear fit
plt.figure()
plt.plot(z_fpr[ok], z_tpr[ok], lw=2, label="Mean Human zROC")
# overlay linear fit
zf_plot = np.linspace(np.nanmin(z_fpr[ok]), np.nanmax(z_fpr[ok]), 200)
plt.plot(zf_plot, slope * zf_plot + intercept, ls="--", label=f"Linear fit (slope={slope:.2f})")
# optional: overlay quadratic fit
plt.plot(zf_plot, a*zf_plot**2 + b*zf_plot + c, ls=":", label=f"Quadratic fit (a={a:.2e})")
plt.xlabel("z(FPR)")
plt.ylabel("z(TPR)")
plt.legend()
plt.title("zROC")
plt.show()

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from scipy.stats import norm

# ---- helpers ----
def z(p, eps=1e-4):
    return norm.ppf(np.clip(p, eps, 1-eps))

def stack_exps(exps):
    # assumes each exp has columns: response, repeat ('true'/'false'), isi
    dfs = []
    for i, e in enumerate(exps):
        d = pd.DataFrame({
            "response": np.asarray(e.response),
            "is_target": np.asarray(e.repeat == "true", dtype=int),
            "isi": np.asarray(e.isi, dtype=float),
            "exp_ix": i
        })
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True)

df = stack_exps(exps)

# In many of your datasets, lures have isi == -1. Keep them as lures for *all* ISI-conditioned ROCs.
LURE_FLAG = (df["is_target"] == 0)  # use all lures

isi_vals = sorted(v for v in df["isi"].unique() if v >= 0)  # exclude -1 (lures)
fpr_grid = np.linspace(0, 1, 200)

per_isi_stats = []
plt_kwargs = dict(alpha=0.9, lw=2)


for isi in isi_vals:
    # targets = repeats with this ISI; lures = all non-repeats
    mask = ((df["is_target"] == 1) & (df["isi"] == isi)) | LURE_FLAG
    d = df.loc[mask].copy()

    y = d["is_target"].to_numpy(dtype=int)
    s = np.array(d["response"].to_numpy(), dtype=np.int32)  # higher => more "old"

    # ROC
    fpr, tpr, _ = roc_curve(y, s, drop_intermediate=False)
    # interpolate for smooth plotting/comparability
    tpr_i = np.interp(fpr_grid, fpr, tpr)

    # zROC
    zf = z(fpr_grid)
    zt = z(tpr_i)
    ok = np.isfinite(zf) & np.isfinite(zt)
    if ok.sum() < 4:
        continue

    lin = np.polyfit(zf[ok], zt[ok], 1)     # slope, intercept
    quad = np.polyfit(zf[ok], zt[ok], 2)    # a, b, c
    slope, intercept = lin
    a, b, c = quad

    per_isi_stats.append({
        "isi": isi,
        "slope": slope,
        "intercept": intercept,
        "quad_a": a, "quad_b": b, "quad_c": c,
        "auc": auc(fpr_grid, tpr_i)
    })

    # plot zROC for this ISI
    zf_plot = np.linspace(zf[ok].min(), zf[ok].max(), 200)
    plt.figure()
    plt.plot(zf[ok], zt[ok], label=f"zROC (ISI={isi:g})", **plt_kwargs)
    plt.plot(zf_plot, slope*zf_plot + intercept, "--", label=f"Linear slope={slope:.2f}")
    plt.plot(zf_plot, a*zf_plot**2 + b*zf_plot + c, ":", label=f"Quad a={a:.2e}")
    plt.xlabel("z(FPR)"); plt.ylabel("z(TPR)")
    plt.title(f"zROC by ISI = {isi:g}")
    plt.legend(); plt.tight_layout(); plt.show()

# # ---- summary plots ----
# stats_df = pd.DataFrame(per_isi_stats).sort_values("isi")

# plt.figure()
# plt.plot(stats_df["isi"], stats_df["slope"], marker="o")
# plt.axhline(1.0, ls="--", lw=1)
# plt.xlabel("ISI"); plt.ylabel("zROC slope")
# plt.title("zROC slope vs ISI"); plt.tight_layout(); plt.show()

# plt.figure()
# plt.plot(stats_df["isi"], stats_df["quad_a"], marker="o")
# plt.axhline(0.0, ls="--", lw=1)
# plt.xlabel("ISI"); plt.ylabel("zROC curvature (quad a)")
# plt.title("zROC curvature vs ISI"); plt.tight_layout(); plt.show()

# print(stats_df)